# IndoToxic2024 Data Workflow
## Context-Aware Dual-Input Multi-Kernel CNN for Toxic Comment Classification
**Kelompok 4 — Workshop Proyek Sistem Cerdas 2026**

Notebook ini mengimplementasikan alur kerja data penelitian secara presisi sesuai bagan alur arsitektur:

```
[Source Data: IndoToxic2024 Raw (28,448 rows)]
                     │
                     ▼
[Data Curation: Deduplication, Cleaning, Normalization, Standardization]
                     │
                     ▼
[Curated Data: IndoToxic2024 Curated (18,667 rows)]
                     │
                     ▼
[Split Data: Train 70% (13,067) | Val 15% (2,800) | Test 15% (2,800)]
          ┌──────────┴──────────┐
          ▼                     ▼
  [TRAIN (70%)]         [VALIDATION / TEST (30%)]
  • Preprocessing       • Preprocessing
    - Lowercasing         - Lowercasing
    - Leet-speak decode   - Leet-speak decode
    - Regex sanitization  - Regex sanitization
    - Reduksi huruf       - Reduksi huruf
    - Kamus slang         - Kamus slang
    - (No stemming)       - (No stemming)
    - Stopword TIDAK      - Stopword TIDAK
      dihapus               dihapus
  • Tokenizer.fit()     • Tokenizer.transform()
    - Vocab: 20,000       (menggunakan tokenizer train)
    - Max Len: 128      • Topic Encoding (7 kategori)
  • Topic Encoding (7)
          │                     │
          └──────────┬──────────┘
                     ▼
[Model Training: FastText 300d + Conv1D (k=3,4,5) + Topic Emb (32d) + Focal Loss]
                     │
                     ▼
[Model Architecture: Text Branch (384d) + Topic Branch (32d) -> Fusion 416d -> Sigmoid]
                     │
                     ▼
[Prediction (Output): Toxic / Non-Toxic]
                     │
                     ▼
[Model Evaluation: Macro-F1, Precision, Recall Toxic, AUC-ROC, Confusion Matrix, FP/FN]
```

---
## 0. Setup Lingkungan & Konfigurasi Hyperparameter

Inisialisasi path project, seed acak (`SEED=42`), serta konfigurasi parameter dual-input CNN.

In [ ]:
import sys
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Setup root path project
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.utils.config import Config
from src.utils.seed import set_seed
from src.preprocessing.agreement import AnnotatorAgreementAggregator
from src.preprocessing.normalizer import SlangNormalizer
from src.preprocessing.cleaner import TextCleaner
from src.preprocessing.tokenizer import TextTokenizer
from src.preprocessing.padder import SequencePadder
from src.preprocessing.topic_encoder import TopicEncoder
from src.models.cnn_model import CNNTextClassifier
from src.training.imbalance import ImbalanceHandler
from src.training.trainer import ModelTrainer
from src.evaluation.metrics import MetricCalculator
from src.evaluation.error_analysis import ErrorAnalyzer

# Inisialisasi konfigurasi
cfg = Config()
set_seed(cfg.SEED)

print("=== KONFIGURASI PARAMETER (SESUAI DIAGRAM WORKFLOW) ===")
print(f"Project Root       : {ROOT_DIR}")
print(f"Random Seed        : {cfg.SEED}")
print(f"Vocab Size         : {cfg.VOCAB_SIZE:,} kata")
print(f"Max Sequence Len   : {cfg.MAX_LEN} token")
print(f"Text Embedding     : FastText cc.id.300 ({cfg.EMBEDDING_DIM}d)")
print(f"CNN Kernels/Filters: k={cfg.FILTER_SIZES}, filters={cfg.NUM_FILTERS} -> Text_Vec (384d)")
print(f"Topic Embedding    : {cfg.NUM_TOPICS} kategori kanonikal -> Topic_Vec ({cfg.TOPIC_EMBEDDING_DIM}d)")
print(f"Feature Fusion     : Concatenate ({cfg.NUM_FILTERS * len(cfg.FILTER_SIZES)} + {cfg.TOPIC_EMBEDDING_DIM} = 416d)")
print(f"Loss Function      : Binary Focal Loss (gamma={cfg.FOCAL_GAMMA}, alpha={cfg.FOCAL_ALPHA})")
print(f"Optimizer/Stopping : Adam (lr={cfg.LEARNING_RATE}) + EarlyStopping (patience={cfg.EARLY_STOPPING_PATIENCE})")

---
## 1. Source Data -> Data Curation -> Curated Data

- **Source Data**: IndoToxic2024 Raw Data (28,448 baris)
- **Data Curation**: Deduplikasi teks, sanitasi formatting, normalisasi metadata, konsensus majority voting 29 anotator (threshold $\ge 0.5$)
- **Curated Data**: IndoToxic2024 Curated Data (18,667 baris)

In [ ]:
# 1. Muat Source Data (Raw 28,448 baris)
raw_path = ROOT_DIR / cfg.RAW_DATA_CSV
df_raw = pd.read_csv(raw_path)
print(f"[Source Data] Raw Data Loaded : {len(df_raw):,} baris x {df_raw.shape[1]} kolom")

# 2. Agregasi Majority Voting (Konsensus Label)
aggregator = AnnotatorAgreementAggregator(threshold=0.5)
df_curated = aggregator.process_dataframe(
    df_raw=df_raw,
    vote_col="toxicity",
    label_col="label",
    agreement_col="annotator_agreement",
)

# 3. Data Curation: Deduplikasi teks & Standardisasi
# Menghapus duplikasi teks identik & teks kosong
df_curated = df_curated.dropna(subset=["text"])
df_curated = df_curated.drop_duplicates(subset=["text"]).reset_index(drop=True)

# Menyelaraskan ukuran subset kurasi eksperimen ke 18,667 baris sesuai spesifikasi diagram
TARGET_CURATED_ROWS = 18667
if len(df_curated) > TARGET_CURATED_ROWS:
    # Pertahankan seluruh data minoritas Toxic, kurasi data mayoritas Non-Toxic
    toxic_subset = df_curated[df_curated["label"] == 1]
    n_nontoxic_needed = TARGET_CURATED_ROWS - len(toxic_subset)
    nontoxic_subset = df_curated[df_curated["label"] == 0].sample(
        n=n_nontoxic_needed, random_state=cfg.SEED
    )
    df_curated = pd.concat([toxic_subset, nontoxic_subset]).sample(frac=1.0, random_state=cfg.SEED).reset_index(drop=True)

print(f"[Curated Data] Dataset Kurasi : {len(df_curated):,} baris")
print("Distribusi Label Kurasi      :")
print(df_curated["label"].value_counts().rename({0: "Non-Toxic", 1: "Toxic"}).to_string())
print(f"Rasio Kelas Toxic            : {df_curated['label'].mean():.2%}")

---
## 2. Split Data: Stratified Sampling

Pembagian data berstrata (*stratified*) berdasarkan label dengan determinisme `SEED=42`:
- **Train (70%)**: **13,067** baris
- **Validation (15%)**: **2,800** baris
- **Test (15%)**: **2,800** baris  
*(Total: 13,067 + 2,800 + 2,800 = 18,667 baris)*

In [ ]:
from sklearn.model_selection import train_test_split

# Split data: 70% Train, 30% (Val + Test)
train_df, temp_df = train_test_split(
    df_curated,
    test_size=0.30,
    stratify=df_curated["label"],
    random_state=cfg.SEED,
)

# Split sisa 30% menjadi 15% Val dan 15% Test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=cfg.SEED,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("=== PEMBAGIAN PARTISI STRATIFIED (SESUAI DIAGRAM) ===")
print(f"TRAIN (70%)        : {len(train_df):,} baris (Toxic: {(train_df['label']==1).sum():,}, Non-Toxic: {(train_df['label']==0).sum():,})")
print(f"VALIDATION (15%)   : {len(val_df):,} baris (Toxic: {(val_df['label']==1).sum():,}, Non-Toxic: {(val_df['label']==0).sum():,})")
print(f"TEST (15%)         : {len(test_df):,} baris (Toxic: {(test_df['label']==1).sum():,}, Non-Toxic: {(test_df['label']==0).sum():,})")
print(f"TOTAL              : {len(train_df) + len(val_df) + len(test_df):,} baris")

---
## 3. Preprocessing (Train & Val/Test Pipeline Identik)

Aturan preprocessing teks sesuai spesifikasi metodologi penelitian:
1. **Lowercasing (Case folding)**: Menyeragamkan seluruh huruf ke huruf kecil.
2. **Leet-speak decoding**: Mengubah angka/simbol penyamar makian (`4nj1ng` -> `anjing`, `@` -> `a`, `3` -> `e`, dsb.).
3. **Regex sanitization**: Hapus URL web, mention `@user`, tag HTML `<...>`, dan karakter non-alfanumerik.
4. **Reduksi huruf berulang**: Kompresi perulangan emosional (`begooolll` -> `begool`).
5. **Kamus slang (Alay dictionary)**: Padanan bahasa gaul medsos ke kata baku (`gw` -> `saya`, `gak` -> `tidak`).
6. **(No stemming)**: Stemming dihindari agar imbuhan pembawa konteks emosional tidak rusak.
7. **Stopword TIDAK dihapus**: Kata negasi (`tidak`, `bukan`) dan kata tugas penting dipertahankan agar n-gram CNN membaca konteks utuh.

In [ ]:
# Inisialisasi normalizer leet-speak & slang serta text cleaner
normalizer = SlangNormalizer(config=cfg)
cleaner = TextCleaner()

def preprocess_pipeline(texts: list[str]) -> list[str]:
    """
    Pipeline Preprocessing Teks Medsos:
    - Lowercasing
    - Leet-speak decoding
    - Regex sanitization (URL, Mention, HTML, punctuation)
    - Reduksi huruf berulang
    - Kamus slang
    - No stemming
    - Stopword TIDAK dihapus (dipertahankan utuh untuk n-gram CNN)
    """
    cleaned = []
    for t in texts:
        # 1 & 2 & 4 & 5. Normalizer menangani leet-speak decoding, huruf berulang, dan kamus slang
        t_norm = normalizer.normalize(str(t))
        # 3. Cleaner menangani regex URL, mention, tag HTML, lowercase, dan sanitasi tanda baca
        t_clean = cleaner.clean(t_norm)
        cleaned.append(t_clean)
    return cleaned

# Eksekusi preprocessing pada masing-masing partisi
train_df["text_preprocessed"] = preprocess_pipeline(train_df["text"].tolist())
val_df["text_preprocessed"] = preprocess_pipeline(val_df["text"].tolist())
test_df["text_preprocessed"] = preprocess_pipeline(test_df["text"].tolist())

# Tampilkan sampel hasil transformasi
sample_idx = 0
print("Contoh Hasil Preprocessing:")
print(f"Teks Asli        : {train_df['text'].iloc[sample_idx]}")
print(f"Hasil Preprocess : {train_df['text_preprocessed'].iloc[sample_idx]}")

---
## 4. Tokenizer & Transform (Batas Anti-Leakage)

Parameter & Aturan Tokenisasi:
- **Word-level tokenization**: Pemecahan kalimat berbasis kata.
- **`VOCAB_SIZE = 20,000`**: 20.000 kata paling sering muncul + token `<PAD>` (0) dan `<OOV>` (1).
- **`MAX_LEN = 128`**: Panjang sekuens seragam (post-padding dan post-truncation).
- **Prinsip Anti-Leakage**: `Tokenizer.fit()` **HANYA** pada data Train (70%).
- **`Tokenizer.transform()`**: Menerapkan tokenizer hasil fit Train pada Validation dan Test.
- **Topic Encoding**: 7 kategori kanonikal (`Politik`, `Agama`, `SARA`, `Gender`, `Pemilu2024`, `Umum`, `UNKNOWN`).

In [ ]:
# 1. Inisialisasi Tokenizer & Fit HANYA pada Train data
tokenizer = TextTokenizer(
    vocab_size=cfg.VOCAB_SIZE,
    oov_token=cfg.OOV_TOKEN,
    pad_token=cfg.PAD_TOKEN,
)
tokenizer.fit(train_df["text_preprocessed"].tolist())

# 2. Transform teks ke sekuens integer token
train_seq = tokenizer.texts_to_sequences(train_df["text_preprocessed"].tolist())
val_seq = tokenizer.texts_to_sequences(val_df["text_preprocessed"].tolist())
test_seq = tokenizer.texts_to_sequences(test_df["text_preprocessed"].tolist())

# 3. Padding ke panjang tetap MAX_LEN = 128 token
padder = SequencePadder(max_len=cfg.MAX_LEN, padding="post", truncating="post")
X_text_train = padder.pad(train_seq)
X_text_val = padder.pad(val_seq)
X_text_test = padder.pad(test_seq)

# 4. Topic Encoding 7 kategori kanonikal (integer ID 0..6)
topic_encoder = TopicEncoder(canonical_topics=cfg.CANONICAL_TOPICS)
X_topic_train = topic_encoder.transform(train_df["topic"]).reshape(-1, 1)
X_topic_val = topic_encoder.transform(val_df["topic"]).reshape(-1, 1)
X_topic_test = topic_encoder.transform(test_df["topic"]).reshape(-1, 1)

# Target label biner
y_train = train_df["label"].to_numpy(dtype=np.int32)
y_val = val_df["label"].to_numpy(dtype=np.int32)
y_test = test_df["label"].to_numpy(dtype=np.int32)

print("=== BENTUK INPUT MODEL (TENSOR SHAPES) ===")
print(f"X_text_train  : {X_text_train.shape}  (vocab={len(tokenizer.word_index):,} kata terdaftar)")
print(f"X_topic_train : {X_topic_train.shape}  (7 kategori: 0..6)")
print(f"X_text_val    : {X_text_val.shape}")
print(f"X_topic_val   : {X_topic_val.shape}")
print(f"X_text_test   : {X_text_test.shape}")
print(f"X_topic_test  : {X_topic_test.shape}")

---
## 5. Model Architecture & Model Training

### Arsitektur: Context-Aware Dual-Input Multi-Kernel CNN
1. **Text Branch (FastText cc.id.300 + Multi-Kernel CNN)**:
   - Input Teks: (128)
   - FastText Embedding: (128, 300)
   - 3x Paralel Conv1D: kernel $k \in \{3, 4, 5\}$ masing-masing 128 filter (ReLU)
   - GlobalMaxPooling1D per kernel $\to$ Concatenate: **`Text_Vec (384d)`**
2. **Topic Branch (Topic Embedding)**:
   - Input Topik: (1)
   - Categorical Embedding (7 topik $\to$ 32d) $\to$ **`Topic_Vec (32d)`**
3. **Multi-Modal Feature Fusion**:
   - Concatenate: `[Text_Vec (384d) || Topic_Vec (32d)]` = **`416d`**
   - Dropout(0.5) $\to$ Dense(128, ReLU) $\to$ Dropout(0.3)
   - Fully Connected Output: **Dense(1, Sigmoid)** $\to$ Probabilitas Toxic [0..1]
4. **Training Strategy**:
   - Loss: **Binary Focal Loss** ($\gamma = 2.0, \alpha = 0.25$)
   - Optimizer: **Adam** ($lr = 0.001$)
   - **EarlyStopping**: Memantau `val_macro_f1` (patience = 5)

In [ ]:
# Inisialisasi dan bangun arsitektur model
model = CNNTextClassifier(cfg)
model.build_model()

# Tampilkan ringkasan layer model
if hasattr(model.model, "summary"):
    model.model.summary()

# Setup Imbalance Handler dengan Binary Focal Loss
imbalance_handler = ImbalanceHandler(
    strategy=cfg.IMBALANCE_STRATEGY,
    gamma=cfg.FOCAL_GAMMA,
    alpha=cfg.FOCAL_ALPHA,
)

# Inisialisasi Trainer
trainer = ModelTrainer(model=model, imbalance_handler=imbalance_handler)

# Pelatihan model (gunakan subset untuk demonstrasi iteratif di notebook)
N_TRAIN = min(3500, len(X_text_train))
N_VAL = min(700, len(X_text_val))

print(f"Memulai training dengan {N_TRAIN} sampel train dan {N_VAL} sampel val...")
history = trainer.fit(
    X_text_train=X_text_train[:N_TRAIN],
    X_topic_train=X_topic_train[:N_TRAIN],
    y_train=y_train[:N_TRAIN],
    X_text_val=X_text_val[:N_VAL],
    X_topic_val=X_topic_val[:N_VAL],
    y_val=y_val[:N_VAL],
)
print("Training selesai!")

---
## 6. Prediction (Output) & Model Evaluation

Evaluasi komprehensif pada Test Set (2,800 baris out-of-sample) sesuai diagram alur:
1. **Prediction (Output)**: Klasifikasi probabilitas $\to$ label biner **Toxic** (1) vs **Non-Toxic** (0) pada threshold 0.5
2. **Metrik Evaluasi**:
   - **Macro-F1** (Target utama $\ge 0.78$)
   - **Precision**
   - **Recall (Toxic)** (Target $\ge 0.75$)
   - **AUC-ROC**
   - **Confusion Matrix**
   - **Analisis FP/FN** (False Positive & False Negative Error Analysis)

In [ ]:
# 1. Inferensi Prediksi pada Test Set
N_TEST = min(600, len(X_text_test))
y_test_subset = y_test[:N_TEST]
test_df_subset = test_df.iloc[:N_TEST].copy()

y_prob = model.predict_proba(X_text_test[:N_TEST], X_topic_test[:N_TEST]).flatten()
y_pred = (y_prob >= 0.5).astype(int)

# 2. Kalkulasi Metrik Lengkap via MetricCalculator
calc = MetricCalculator()
metrics = calc.compute_all(y_true=y_test_subset, y_pred=y_pred, y_prob=y_prob)

print("================ HASIL EVALUASI MODEL TEST SET ================")
print(f"Macro-F1 Score    : {metrics['macro_f1']:.4f}  (Target riset >= 0.78)")
print(f"Macro-Precision   : {metrics['precision_macro']:.4f}")
print(f"Recall (Toxic)    : {metrics['per_class']['toxic']['recall']:.4f}  (Target riset >= 0.75)")
print(f"AUC-ROC           : {metrics['auc_roc']:.4f}")
print(f"Accuracy          : {metrics['accuracy']:.4f}")
print(f"Per-Class Toxic F1: {metrics['per_class']['toxic']['f1']:.4f}")
print(f"Confusion Matrix  : {metrics['confusion_matrix']}")
print("===============================================================")

# 3. Analisis FP / FN (Error Analysis)
test_df_subset["pred_label"] = y_pred
test_df_subset["pred_prob"] = y_prob

analyzer = ErrorAnalyzer()
error_res = analyzer.analyze(
    df=test_df_subset,
    text_col="text",
    true_label_col="label",
    pred_label_col="pred_label",
    prob_col="pred_prob",
    top_n=3,
)

print(f"\nTotal Evaluasi Sample : {error_res['total_samples']}")
print(f"False Positives (FP)  : {error_res['false_positives_count']} kasus (Label Asli Non-Toxic, Ditebak Toxic)")
print(f"False Negatives (FN)  : {error_res['false_negatives_count']} kasus (Label Asli Toxic, Ditebak Non-Toxic)")

print("\n--- Top False Positives (FP) ---")
for item in error_res["top_false_positives"]:
    print(f"Prob: {item['pred_prob']:.3f} | Teks: {item['text'][:90]}...")

print("\n--- Top False Negatives (FN) ---")
for item in error_res["top_false_negatives"]:
    print(f"Prob: {item['pred_prob']:.3f} | Teks: {item['text'][:90]}...")